# Advanced Object-Oriented Programming in Python

## Table of Contents
1. [Metaclasses (type, custom metaclasses)](#metaclasses)
2. [Abstract Base Classes in Depth](#abc-in-depth)
3. [Descriptors in Depth](#descriptors)
4. [MRO (Method Resolution Order)](#mro)
5. [SOLID Principles](#solid-principles)
6. [Design Patterns](#design-patterns)
7. [Mixins](#mixins)
8. [Monkey Patching](#monkey-patching)
9. [Proxies and Delegation](#proxies-and-delegation)
10. [LeetCode Problems](#leetcode-problems)

## 1. Metaclasses (type, custom metaclasses) <a id="metaclasses"></a>

In Python, **everything is an object** — including classes. Classes are instances of **metaclasses**.

### Key Points:
- The default metaclass is `type`
- Metaclasses control class creation
- You can create custom metaclasses by subclassing `type`
- The `__new__` and `__init__` methods of a metaclass control class creation

### Example:

In [ ]:
# Everything is an object - even classes
class MyClass:
    pass

print(f"Type of MyClass: {type(MyClass)}")
print(f"Type of instance: {type(MyClass())}")

# Creating a class using type() directly
DynamicClass = type('DynamicClass', (object,), {
    'x': 10,
    'greet': lambda self: f"Hello from dynamic class! x={self.x}"
})

obj = DynamicClass()
print(f"Dynamic class: {obj.greet()}")

# Custom metaclass
class SingletonMeta(type):
    """Metaclass that creates Singleton classes."""
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class DatabaseConnection(metaclass=SingletonMeta):
    def __init__(self, host='localhost'):
        self.host = host
    
    def connect(self):
        return f"Connected to {self.host}"

# Test Singleton
db1 = DatabaseConnection('db1.example.com')
db2 = DatabaseConnection('db2.example.com')

print(f"Same instance: {db1 is db2}")
print(f"Host: {db1.host}")  # Still db1.example.com

## 2. Abstract Base Classes in Depth <a id="abc-in-depth"></a>

ABCs provide more than just abstract methods — they support **virtual subclassing** and **registering** classes.

### Key Points:
- Use `ABC` as a base class or use `ABCMeta` as a metaclass
- `@abstractmethod` marks methods that must be implemented
- `@abstractproperty` for abstract properties
- `register()` allows registering classes as virtual subclasses
- `__subclasshook__` can define custom subclass checks

### Example:

In [ ]:
from abc import ABC, ABCMeta, abstractmethod, abstractproperty

class MyABC(ABC):
    """Abstract base class with registration support."""
    
    @abstractmethod
    def process(self):
        pass

# Virtual subclassing - no inheritance needed
class MyProcessor:
    def process(self):
        return "Processing data"

MyABC.register(MyProcessor)

print(f"Is MyProcessor a subclass of MyABC? {issubclass(MyProcessor, MyABC)}")
print(f"Is instance a MyABC? {isinstance(MyProcessor(), MyABC)}")

# Custom subclass hook
class ContainerLike(ABCMeta):
    """Metaclass that checks for container-like methods."""
    def __subclasshook__(cls, subclass):
        if cls is ContainerLike:
            has_methods = all(
                hasattr(subclass, method) for method in ['__getitem__', '__len__', '__contains__']
            )
            if has_methods:
                return True
        return NotImplemented

class Container(metaclass=ContainerLike):
    pass

class MyList:
    def __getitem__(self, index):
        return index
    def __len__(self):
        return 0
    def __contains__(self, item):
        return False

print(f"\nMyList is Container-like: {issubclass(MyList, Container)}")
print(f"MyList instance is Container: {isinstance(MyList(), Container)}")

## 3. Descriptors in Depth <a id="descriptors"></a>

A **descriptor** is a protocol that allows you to customize attribute access.

### Key Points:
- Implement `__get__`, `__set__`, and `__delete__` methods
- **Data descriptors**: Define `__set__` or `__delete__` (have priority over instance dict)
- **Non-data descriptors**: Only define `__get__` (can be overridden by instance dict)
- Used internally by `property`, `classmethod`, `staticmethod`, and `function`

### Example:

In [ ]:
class Descriptor:
    """A data descriptor that logs attribute access."""
    
    def __init__(self, name):
        self.name = name
        self._value = None
    
    def __get__(self, instance, owner):
        if instance is None:
            return self
        print(f"Getting {self.name}")
        return self._value
    
    def __set__(self, instance, value):
        print(f"Setting {self.name} to {value}")
        self._value = value
    
    def __delete__(self, instance):
        print(f"Deleting {self.name}")
        self._value = None

class NonDataDescriptor:
    """A non-data descriptor (only __get__)."""
    
    def __init__(self, value):
        self.value = value
    
    def __get__(self, instance, owner):
        return self.value

class MyClass:
    # Data descriptor - has priority over instance __dict__
    data_attr = Descriptor('data_attr')
    
    # Non-data descriptor - can be overridden by instance __dict__
    non_data_attr = NonDataDescriptor('non_data_value')

obj = MyClass()

# Data descriptor
obj.data_attr = 42
print(f"Value: {obj.data_attr}")
del obj.data_attr

# Non-data descriptor
print(f"Non-data: {obj.non_data_attr}")
obj.non_data_attr = 'overridden'
print(f"After override: {obj.non_data_attr}")

# Property is a data descriptor
class Temperature:
    def __init__(self, celsius=0):
        self._celsius = celsius
    
    @property
    def celsius(self):
        return self._celsius
    
    @celsius.setter
    def celsius(self, value):
        self._celsius = value

temp = Temperature(25)
print(f"\nTemperature: {temp.celsius}")
temp.celsius = 30
print(f"Updated: {temp.celsius}")

## 4. MRO (Method Resolution Order) <a id="mro"></a>

MRO is the order in which Python looks for methods in a class hierarchy.

### Key Points:
- Python uses the **C3 linearization** algorithm
- Accessed via `ClassName.__mro__` or `ClassName.mro()`
- Follows the **L(O)DFS** (Left, then Oldest, then Depth, then Siblings) rule
- Ensures a consistent and predictable method resolution order

### Example:

In [ ]:
class A:
    def method(self):
        return "A"

class B(A):
    pass

class C(A):
    def method(self):
        return "C"

class D(B, C):
    pass

print(f"MRO of D: {[cls.__name__ for cls in D.__mro__]}")
print(f"D().method() returns: {D().method()}")  # C (not A)

# Diamond inheritance
class Animal:
    def speak(self):
        return "Some sound"

class Mammal(Animal):
    def speak(self):
        return super().speak() + " from Mammal"

class Winged(Animal):
    def speak(self):
        return super().speak() + " from Winged"

class Bat(Mammal, Winged):
    def speak(self):
        return super().speak() + " from Bat"

print(f"\nMRO of Bat: {[cls.__name__ for cls in Bat.__mro__]}")
print(f"Bat().speak(): {Bat().speak()}")

# C3 Linearization example
class X: pass
class Y: pass
class Z: pass
class A1(X, Y): pass
class B1(Y, Z): pass
class C1(A1, B1, Z): pass

print(f"\nMRO of C1: {[cls.__name__ for cls in C1.__mro__]}")

## 5. SOLID Principles <a id="solid-principles"></a>

SOLID is an acronym for five design principles in object-oriented programming.

### S - Single Responsibility Principle (SRP)
A class should have only one reason to change.

### O - Open/Closed Principle (OCP)
Software entities should be open for extension but closed for modification.

### L - Liskov Substitution Principle (LSP)
Objects of a superclass should be replaceable with objects of its subclasses.

### I - Interface Segregation Principle (ISP)
Clients should not be forced to depend on interfaces they don't use.

### D - Dependency Inversion Principle (DIP)
Depend on abstractions, not concretions.

In [ ]:
# S - Single Responsibility Principle
class Invoice:
    def __init__(self, amount):
        self.amount = amount

class InvoicePrinter:
    def print(self, invoice):
        return f"Invoice: ${invoice.amount}"

class InvoiceRepository:
    def save(self, invoice):
        return f"Saved invoice with amount ${invoice.amount}"

# O - Open/Closed Principle
from abc import ABC, abstractmethod

class Discount(ABC):
    @abstractmethod
    def apply(self, amount):
        pass

class PercentageDiscount(Discount):
    def __init__(self, percentage):
        self.percentage = percentage
    def apply(self, amount):
        return amount * (1 - self.percentage / 100)

class FixedDiscount(Discount):
    def __init__(self, amount):
        self.amount = amount
    def apply(self, amount):
        return amount - self.amount

class ShoppingCart:
    def __init__(self):
        self.items = []
    def add_item(self, price):
        self.items.append(price)
    def total(self, discount=None):
        total = sum(self.items)
        if discount:
            return discount.apply(total)
        return total

# L - Liskov Substitution Principle
class Rectangle:
    def __init__(self, width, height):
        self._width = width
        self._height = height
    @property
    def width(self):
        return self._width
    @property
    def height(self):
        return self._height
    def area(self):
        return self._width * self._height

class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)

# I - Interface Segregation Principle
class Printer(ABC):
    @abstractmethod
    def print(self, document):
        pass

class Scanner(ABC):
    @abstractmethod
    def scan(self, document):
        pass

class SimplePrinter(Printer):
    def print(self, document):
        return f"Printing: {document}"

class MultiFunctionDevice(Printer, Scanner):
    def print(self, document):
        return f"Printing: {document}"
    def scan(self, document):
        return f"Scanning: {document}"

# D - Dependency Inversion Principle
class Database(ABC):
    @abstractmethod
    def save(self, data):
        pass

class MySQLDatabase(Database):
    def save(self, data):
        return f"Saved to MySQL: {data}"

class MongoDBDatabase(Database):
    def save(self, data):
        return f"Saved to MongoDB: {data}"

class UserService:
    def __init__(self, database: Database):
        self.database = database
    def create_user(self, name):
        return self.database.save(f"User: {name}")

# Test all principles
invoice = Invoice(100)
printer = InvoicePrinter()
print(printer.print(invoice))

cart = ShoppingCart()
cart.add_item(50)
cart.add_item(30)
discount = PercentageDiscount(10)
print(f"Total with discount: ${cart.total(discount)}")

service = UserService(MySQLDatabase())
print(service.create_user('Alice'))
service = UserService(MongoDBDatabase())
print(service.create_user('Bob'))

## 6. Design Patterns <a id="design-patterns"></a>

Design patterns are reusable solutions to common problems in software design.

### Creational Patterns:
- **Singleton**: Ensure only one instance exists
- **Factory Method**: Create objects without specifying the exact class
- **Abstract Factory**: Create families of related objects
- **Builder**: Construct complex objects step by step
- **Prototype**: Create new objects by cloning existing ones

### Structural Patterns:
- **Adapter**: Convert one interface to another
- **Bridge**: Decouple abstraction from implementation
- **Composite**: Compose objects into tree structures
- **Decorator**: Add behavior to objects dynamically
- **Facade**: Provide a simplified interface

### Behavioral Patterns:
- **Observer**: Notify dependent objects of state changes
- **Strategy**: Define a family of algorithms
- **Command**: Encapsulate requests as objects
- **Template Method**: Define the skeleton of an algorithm
- **Iterator**: Access elements sequentially

In [ ]:
# Creational: Singleton
class Singleton:
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

# Creational: Factory Method
class VehicleFactory:
    @staticmethod
    def create_vehicle(type_):
        if type_ == 'car':
            return Car()
        elif type_ == 'bike':
            return Bike()
        return None

class Car:
    def drive(self):
        return "Driving a car"

class Bike:
    def drive(self):
        return "Riding a bike"

# Creational: Builder
class HouseBuilder:
    def __init__(self):
        self.house = House()
    def build_walls(self):
        self.house.walls = 4
        return self
    def build_doors(self):
        self.house.doors = 1
        return self
    def build_windows(self):
        self.house.windows = 2
        return self
    def get_result(self):
        return self.house

class House:
    def __init__(self):
        self.walls = 0
        self.doors = 0
        self.windows = 0
    def __str__(self):
        return f"House(walls={self.walls}, doors={self.doors}, windows={self.windows})"

# Structural: Decorator
class Coffee:
    def cost(self):
        return 5

class MilkDecorator:
    def __init__(self, coffee):
        self._coffee = coffee
    def cost(self):
        return self._coffee.cost() + 2

class SugarDecorator:
    def __init__(self, coffee):
        self._coffee = coffee
    def cost(self):
        return self._coffee.cost() + 1

# Behavioral: Observer
class Subject:
    def __init__(self):
        self._observers = []
        self._state = None
    def attach(self, observer):
        self._observers.append(observer)
    def set_state(self, state):
        self._state = state
        for obs in self._observers:
            obs.update(state)

class Observer:
    def __init__(self, name):
        self.name = name
    def update(self, state):
        print(f"{self.name} received state: {state}")

# Behavioral: Strategy
class SortStrategy(ABC):
    @abstractmethod
    def sort(self, data):
        pass

class QuickSort(SortStrategy):
    def sort(self, data):
        return sorted(data)  # Simplified

class BubbleSort(SortStrategy):
    def sort(self, data):
        return sorted(data)  # Simplified

class Context:
    def __init__(self, strategy: SortStrategy):
        self._strategy = strategy
    def execute_sort(self, data):
        return self._strategy.sort(data)

# Test patterns
s1 = Singleton()
s2 = Singleton()
print(f"Singleton same instance: {s1 is s2}")

vehicle = VehicleFactory.create_vehicle('car')
print(f"Factory: {vehicle.drive()}")

house = HouseBuilder().build_walls().build_doors().build_windows().get_result()
print(f"Builder: {house}")

coffee = SugarDecorator(MilkDecorator(Coffee()))
print(f"Decorator: Coffee costs ${coffee.cost()}")

subject = Subject()
subject.attach(Observer('A'))
subject.attach(Observer('B'))
subject.set_state('active')

context = Context(QuickSort())
print(f"Strategy: {context.execute_sort([3, 1, 2])}")

## 7. Mixins <a id="mixins"></a>

A **mixin** is a class that provides methods to other classes but is not meant to be instantiated on its own.

### Key Points:
- Mixins provide specific functionality that can be reused across multiple classes
- They don't have their own state (or if they do, it's minimal)
- Use multiple inheritance to combine mixins
- The name usually ends with `Mixin` suffix

### Example:

In [ ]:
class JSONMixin:
    """Mixin to add JSON serialization."""
    import json
    def to_json(self):
        return self.json.dumps(self.__dict__)

class ReprMixin:
    """Mixin to add detailed repr."""
    def __repr__(self):
        attrs = ', '.join(f'{k}={v!r}' for k, v in self.__dict__.items())
        return f"{self.__class__.__name__}({attrs})"

class TimestampMixin:
    """Mixin to add timestamp functionality."""
    from datetime import datetime
    def timestamp(self):
        return self.datetime.now().isoformat()

class User(JSONMixin, ReprMixin, TimestampMixin):
    """User class with JSON, repr, and timestamp functionality via mixins."""
    def __init__(self, name, email):
        self.name = name
        self.email = email

user = User('Alice', 'alice@example.com')
print(f"Repr: {repr(user)}")
print(f"JSON: {user.to_json()}")
print(f"Timestamp: {user.timestamp()}")

# Another example: Logging mixin
class LoggingMixin:
    """Mixin to add logging to any class."""
    def log(self, message):
        print(f"[{self.__class__.__name__}] {message}")

class DataProcessor(LoggingMixin):
    def process(self, data):
        self.log(f"Processing {len(data)} items")
        result = [item.upper() for item in data]
        self.log(f"Processed {len(result)} items")
        return result

processor = DataProcessor()
result = processor.process(['hello', 'world'])
print(f"Result: {result}")

# Mixin with abstract methods
class ComparableMixin:
    """Mixin that adds comparison operators based on a key function."""
    def _compare_key(self):
        raise NotImplementedError
    def __lt__(self, other):
        return self._compare_key() < other._compare_key()
    def __eq__(self, other):
        return self._compare_key() == other._compare_key()

class Product(ComparableMixin):
    def __init__(self, name, price):
        self.name = name
        self.price = price
    def _compare_key(self):
        return self.price

products = [Product('A', 100), Product('B', 50), Product('C', 200)]
sorted_products = sorted(products)
print(f"Sorted by price: {[p.name for p in sorted_products]}")

## 8. Monkey Patching <a id="monkey-patching"></a>

Monkey patching is the practice of modifying code at runtime by adding, replacing, or deleting attributes.

### Key Points:
- Can modify classes and modules at runtime
- Useful for debugging, testing, and extending third-party libraries
- Can be dangerous if not done carefully
- Should be used sparingly in production code

### Example:

In [ ]:
# Monkey patching a class method
class Calculator:
    def add(self, a, b):
        return a + b

calc = Calculator()
print(f"Original: {calc.add(2, 3)}")

# Replace the method at runtime
def multiply(self, a, b):
    return a * b

Calculator.add = multiply
print(f"After patching: {calc.add(2, 3)}")

# Monkey patching instance method
def subtract(self, a, b):
    return a - b

import types
calc.subtract = types.MethodType(subtract, calc)
print(f"Instance method: {calc.subtract(5, 3)}")

# Monkey patching a module
import math

original_sqrt = math.sqrt

def patched_sqrt(x):
    if x < 0:
        raise ValueError("Cannot take square root of negative number")
    return original_sqrt(x)

math.sqrt = patched_sqrt
print(f"Patched sqrt: {math.sqrt(16)}")

try:
    math.sqrt(-1)
except ValueError as e:
    print(f"Error: {e}")

# Restore original
math.sqrt = original_sqrt
print(f"Restored sqrt: {math.sqrt(16)}")

# Monkey patching for testing
class APIClient:
    def fetch_data(self):
        # Imagine this makes a real API call
        return "Real API data"

def mock_fetch_data(self):
    return "Mocked API data"

APIClient.fetch_data = mock_fetch_data
client = APIClient()
print(f"Mocked: {client.fetch_data()}")

## 9. Proxies and Delegation <a id="proxies-and-delegation"></a>

A **proxy** is an object that controls access to another object. **Delegation** is when an object delegates its behavior to another object.

### Types of Proxies:
- **Virtual Proxy**: Creates expensive objects on demand
- **Protection Proxy**: Controls access based on permissions
- **Smart Reference**: Adds behavior when an object is accessed

### Example:

In [ ]:
# Virtual Proxy - lazy initialization
class ExpensiveObject:
    def __init__(self):
        print("Expensive object created")
    def process(self):
        return "Processing data"

class VirtualProxy:
    def __init__(self):
        self._obj = None
    def process(self):
        if self._obj is None:
            self._obj = ExpensiveObject()
        return self._obj.process()

print("Creating proxy...")
proxy = VirtualProxy()
print("Proxy created, no expensive object yet")
print(proxy.process())
print(proxy.process())  # Object already created

# Protection Proxy - access control
class SensitiveData:
    def __init__(self, data):
        self._data = data
    def read(self):
        return self._data
    def write(self, data):
        self._data = data

class ProtectionProxy:
    def __init__(self, data, role):
        self._obj = SensitiveData(data)
        self._role = role
    def read(self):
        if self._role in ('admin', 'user'):
            return self._obj.read()
        raise PermissionError("Read access denied")
    def write(self, data):
        if self._role == 'admin':
            self._obj.write(data)
        else:
            raise PermissionError("Write access denied")

admin_proxy = ProtectionProxy('secret', 'admin')
user_proxy = ProtectionProxy('secret', 'user')
guest_proxy = ProtectionProxy('secret', 'guest')

print(f"Admin read: {admin_proxy.read()}")
admin_proxy.write('new secret')
print(f"Admin after write: {admin_proxy.read()}")

print(f"User read: {user_proxy.read()}")
try:
    user_proxy.write('hacked')
except PermissionError as e:
    print(f"User write denied: {e}")

try:
    guest_proxy.read()
except PermissionError as e:
    print(f"Guest read denied: {e}")

# Delegation pattern
class Engine:
    def start(self):
        return "Engine started"
    def stop(self):
        return "Engine stopped"

class Car:
    def __init__(self):
        self._engine = Engine()
    # Delegate to engine
    def start(self):
        return self._engine.start()
    def stop(self):
        return self._engine.stop()
    def drive(self):
        return "Car is driving"

car = Car()
print(f"\nDelegation: {car.start()}")
print(f"Delegation: {car.drive()}")
print(f"Delegation: {car.stop()}")

## 10. LeetCode Problems <a id="leetcode-problems"></a>

Now let's apply advanced OOP concepts to solve some LeetCode problems.

### Problem 1: Design In-Memory File System (LeetCode #588)

Design a file system that supports:
- Creating directories and files
- Writing and reading file content
- Listing directory contents

Implement the `FileSystem` class:
- `ls(path)` returns the contents of the directory or file name
- `mkdir(path)` creates a new directory
- `addContentToFile(filePath, content)` adds content to a file
- `readContentFromFile(filePath)` returns the file content

In [ ]:
class FileSystem:
    """In-memory file system with directories and files."""
    
    class _Node:
        """Base node for filesystem entries."""
        __slots__ = ('name', 'is_file')
        
        def __init__(self, name, is_file=False):
            self.name = name
            self.is_file = is_file
    
    class _Directory(_Node):
        """Directory node containing child nodes."""
        __slots__ = ('children',)
        
        def __init__(self, name):
            super().__init__(name, is_file=False)
            self.children = {}
    
    class _File(_Node):
        """File node containing content."""
        __slots__ = ('content',)
        
        def __init__(self, name):
            super().__init__(name, is_file=True)
            self.content = []
    
    def __init__(self):
        self._root = self._Directory('')
    
    def _navigate(self, path):
        """Navigate to a node at the given path."""
        if path == '/':
            return self._root
        
        parts = path.strip('/').split('/')
        current = self._root
        
        for part in parts:
            if part not in current.children:
                return None
            current = current.children[part]
        
        return current
    
    def _navigate_to_parent(self, path):
        """Navigate to the parent directory of the given path."""
        parent_path = '/'.join(path.strip('/').split('/')[:-1])
        if parent_path:
            return self._navigate('/' + parent_path)
        return self._root
    
    def ls(self, path: str) -> list:
        """List contents of a directory or return file name."""
        node = self._navigate(path)
        if node is None:
            return []
        
        if node.is_file:
            return [node.name]
        
        return sorted(node.children.keys())
    
    def mkdir(self, path: str) -> None:
        """Create a new directory."""
        parts = path.strip('/').split('/')
        current = self._root
        
        for part in parts:
            if part not in current.children:
                current.children[part] = self._Directory(part)
            current = current.children[part]
    
    def addContentToFile(self, filePath: str, content: str) -> None:
        """Add content to a file, creating it if necessary."""
        parent = self._navigate_to_parent(filePath)
        filename = filePath.strip('/').split('/')[-1]
        
        if filename not in parent.children:
            parent.children[filename] = self._File(filename)
        
        parent.children[filename].content.append(content)
    
    def readContentFromFile(self, filePath: str) -> str:
        """Read content from a file."""
        node = self._navigate(filePath)
        if node and node.is_file:
            return ''.join(node.content)
        return ''

# Test the FileSystem
fs = FileSystem()
fs.mkdir('/a/b/c')
fs.addContentToFile('/a/b/c/d', 'hello world')
print(f"ls /a/b/c: {fs.ls('/a/b/c')}")
print(f"ls /a: {fs.ls('/a')}")
print(f"Read /a/b/c/d: {fs.readContentFromFile('/a/b/c/d')}")
fs.addContentToFile('/a/b/c/d', ' again')
print(f"Read after append: {fs.readContentFromFile('/a/b/c/d')}")

### Problem 2: Design Search Autocomplete System (LeetCode #642)

Design a search autocomplete system that suggests the top 3 historical search queries.

Implement the `AutocompleteSystem` class:
- `input(c)` processes a character and returns top 3 suggestions
- When input is '#', the current query is saved and the system resets

Suggestions are sorted by frequency (descending), then by ASCII order.

In [ ]:
from collections import defaultdict
import heapq

class AutocompleteSystem:
    """Autocomplete system using a trie with frequency tracking."""
    
    class _TrieNode:
        __slots__ = ('children', 'is_end', 'sentences')
        
        def __init__(self):
            self.children = {}
            self.is_end = False
            self.sentences = set()
    
    def __init__(self, sentences: list, times: list):
        self._root = self._TrieNode()
        self._freq = defaultdict(int)
        self._current_query = []
        
        for sentence, time in zip(sentences, times):
            self._freq[sentence] = time
            self._insert_to_trie(sentence)
    
    def _insert_to_trie(self, sentence):
        """Insert a sentence into the trie."""
        node = self._root
        for char in sentence:
            if char not in node.children:
                node.children[char] = self._TrieNode()
            node = node.children[char]
            node.sentences.add(sentence)
        node.is_end = True
    
    def input(self, c: str) -> list:
        """Process an input character and return top 3 suggestions."""
        if c == '#':
            query = ''.join(self._current_query)
            self._freq[query] += 1
            if self._freq[query] == 1:
                self._insert_to_trie(query)
            self._current_query = []
            return []
        
        self._current_query.append(c)
        
        # Navigate to the node for current query
        node = self._root
        for char in self._current_query:
            if char not in node.children:
                return []
            node = node.children[char]
        
        # Get top 3 by frequency, then by ASCII order
        candidates = list(node.sentences)
        candidates.sort(key=lambda s: (-self._freq[s], s))
        
        return candidates[:3]

# Test the AutocompleteSystem
autocomplete = AutocompleteSystem(['i love you', 'island', 'ironman', 'i love leetcode'], [5, 3, 2, 2])

print(f"Input 'i': {autocomplete.input('i')}")
print(f"Input ' ' (space): {autocomplete.input(' ')}")
print(f"Input 'a': {autocomplete.input('a')}")
print(f"Input 'd': {autocomplete.input('d')}")
print(f"Input '#': {autocomplete.input('#')}")
print(f"Input 'i': {autocomplete.input('i')}")
print(f"Input 'r': {autocomplete.input('r')}")
print(f"Input '#': {autocomplete.input('#')}")

### Problem 3: Design an Expression Tree (LeetCode #1628)

Design an expression tree that can evaluate arithmetic expressions.

You need to implement:
- `Node` class: Abstract node with `evaluate()` method
- `NumericNode`: Leaf node with a number
- `BinaryNode`: Internal node with left, right children and an operator

The expression is given in postfix notation.

In [ ]:
from abc import ABC, abstractmethod
import operator

class Node(ABC):
    """Abstract base class for expression tree nodes."""
    
    @abstractmethod
    def evaluate(self) -> int:
        pass

class NumericNode(Node):
    """Leaf node containing a numeric value."""
    __slots__ = ('_value',)
    
    def __init__(self, value: int):
        self._value = value
    
    def evaluate(self) -> int:
        return self._value

class BinaryNode(Node):
    """Internal node with an operator and two children."""
    __slots__ = ('_left', '_right', '_op')
    
    _OPERATORS = {
        '+': operator.add,
        '-': operator.sub,
        '*': operator.mul,
        '/': operator.floordiv,
    }
    
    def __init__(self, left: Node, right: Node, op: str):
        self._left = left
        self._right = right
        self._op = op
    
    def evaluate(self) -> int:
        left_val = self._left.evaluate()
        right_val = self._right.evaluate()
        return self._OPERATORS[self._op](left_val, right_val)

def build_tree(postfix: list) -> Node:
    """Build an expression tree from postfix notation."""
    stack = []
    operators = {'+', '-', '*', '/'}
    
    for token in postfix:
        if token in operators:
            right = stack.pop()
            left = stack.pop()
            node = BinaryNode(left, right, token)
            stack.append(node)
        else:
            node = NumericNode(int(token))
            stack.append(node)
    
    return stack[0]

# Test the Expression Tree
# Expression: (2 + 3) * 4 = 20
# Postfix: 2 3 + 4 *
postfix = ['2', '3', '+', '4', '*']
tree = build_tree(postfix)
print(f"Expression: (2 + 3) * 4 = {tree.evaluate()}")

# Expression: 10 - 2 * 3 = 4
# Postfix: 10 2 3 * -
postfix = ['10', '2', '3', '*', '-']
tree = build_tree(postfix)
print(f"Expression: 10 - 2 * 3 = {tree.evaluate()}")

# Expression: (10 + 5) / 3 * 2 = 10
# Postfix: 10 5 + 3 / 2 *
postfix = ['10', '5', '+', '3', '/', '2', '*']
tree = build_tree(postfix)
print(f"Expression: (10 + 5) / 3 * 2 = {tree.evaluate()}")

### Problem 4: Design Movie Rental System (LeetCode #1912)

Design a movie rental system that tracks movies and their rental prices.

Implement the `MovieRentingSystem` class:
- `search(movie)` returns the 5 cheapest available copies of a movie
- `rent(shop, movie)` rents a movie from a shop
- `drop(shop, movie)` returns a rented movie
- `report()` returns the 5 cheapest rented movies

In [ ]:
from collections import defaultdict
import heapq

class MovieRentingSystem:
    """Movie rental system tracking prices and rentals."""
    
    def __init__(self, n: int, entries: list):
        """
        Initialize the system.
        :param n: Number of shops
        :param entries: List of [shop, movie, price]
        """
        self._prices = {}  # (shop, movie) -> price
        self._available = defaultdict(set)  # movie -> set of (price, shop)
        self._rented = set()  # set of (shop, movie)
        
        for shop, movie, price in entries:
            self._prices[(shop, movie)] = price
            self._available[movie].add((price, shop))
    
    def search(self, movie: int) -> list:
        """Return the 5 cheapest available copies of a movie."""
        if movie not in self._available:
            return []
        
        # Get available copies sorted by price, then shop
        available = sorted(self._available[movie])
        result = []
        for price, shop in available:
            result.append(shop)
            if len(result) == 5:
                break
        return result
    
    def rent(self, shop: int, movie: int) -> None:
        """Rent a movie from a shop."""
        price = self._prices.get((shop, movie))
        if price is not None:
            self._available[movie].discard((price, shop))
            self._rented.add((shop, movie))
    
    def drop(self, shop: int, movie: int) -> None:
        """Return a rented movie."""
        if (shop, movie) in self._rented:
            price = self._prices[(shop, movie)]
            self._rented.remove((shop, movie))
            self._available[movie].add((price, shop))
    
    def report(self) -> list:
        """Return the 5 cheapest rented movies."""
        rented_list = []
        for shop, movie in self._rented:
            price = self._prices[(shop, movie)]
            rented_list.append((price, shop, movie))
        
        rented_list.sort()
        result = []
        for price, shop, movie in rented_list:
            result.append([shop, movie])
            if len(result) == 5:
                break
        return result

# Test the MovieRentingSystem
entries = [
    [0, 1, 5], [0, 2, 6], [0, 3, 7],
    [1, 1, 4], [1, 2, 8], [1, 3, 9],
    [2, 1, 5], [2, 2, 3], [2, 3, 10]
]

system = MovieRentingSystem(3, entries)

print(f"Search movie 1: {system.search(1)}")
print(f"Search movie 2: {system.search(2)}")

system.rent(1, 1)
system.rent(2, 2)

print(f"Search movie 1 after rent: {system.search(1)}")
print(f"Report: {system.report()}")

system.drop(1, 1)
print(f"Search movie 1 after drop: {system.search(1)}")
print(f"Report after drop: {system.report()}")

## Summary

In this notebook, we covered advanced OOP concepts in Python:

1. **Metaclasses**: Customizing class creation with `type` and custom metaclasses
2. **ABCs in Depth**: Virtual subclassing and custom subclass hooks
3. **Descriptors**: Customizing attribute access with `__get__`, `__set__`, `__delete__`
4. **MRO**: Understanding Python's C3 linearization algorithm
5. **SOLID Principles**: All five principles with practical examples
6. **Design Patterns**: Creational, structural, and behavioral patterns
7. **Mixins**: Reusable functionality through multiple inheritance
8. **Monkey Patching**: Runtime modification of classes and modules
9. **Proxies and Delegation**: Controlling access and delegating behavior
10. **Practical Applications**: Solving LeetCode problems with advanced OOP

These concepts provide a deep understanding of Python's OOP capabilities and enable you to write more maintainable, flexible, and powerful code.